# V5W_05 — Replica dei fenotipi C0/C1 (5 parole)

Adattato da **EEG_16** / **EEG_16b**. I 41 soggetti 5words sono **nuovi** → questa è una **replica indipendente**: i due fenotipi (fronto-motor C0 / fronto-occipital C1) riemergono su una coorte diversa?

Pipeline: feature per-soggetto dai grafi (media `adj` e `H@Hᵀ` → triangolo sup.) → PCA(20) + KMeans(k=2) → silhouette + ARI grafo-vs-ipergrafo + mappa differenza C1−C0 con hub. NB: non si può fare ARI fra coorti (soggetti diversi); la replica si valuta su struttura (silhouette), consistenza (ARI interno) e **anatomia degli hub**.

**Env: `daniele_311`**.

## §1 — Config + montage

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w05')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'v5w05'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

N_CHAN = 61
METRIC = 'abs_pcc'
RANDOM_SEED = 42
triu_idx = np.triu_indices(N_CHAN, k=1)
HG_ROOT = project_root / 'data' / '5words_subjects' / 'graphs' / f'hypergraphs_pruned_{METRIC}'
assert HG_ROOT.exists(), f'Grafi non trovati: {HG_ROOT} — esegui prima V5W_02'

# Montage per topomap (stesso di EEG_40)
import mne
ELOC = project_root / 'src' / 'io' / 'ebneuro.locs'
_RENAME = {'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}; _BAD = {'A1','A2'}
_mont = mne.channels.read_custom_montage(str(ELOC), coord_frame='head')
_pos = _mont.get_positions()['ch_pos']
CHAN_NAMES = [_RENAME.get(c, c) for c in _mont.ch_names if c not in _BAD][:N_CHAN]
CH_POS = {_RENAME.get(c, c): _pos[c] for c in _mont.ch_names if c not in _BAD}
def make_topo_info(ch=CHAN_NAMES):
    info = mne.create_info(list(ch), sfreq=256, ch_types='eeg')
    info.set_montage(mne.channels.make_dig_montage(ch_pos={c: CH_POS[c] for c in ch}, coord_frame='head'), on_missing='warn')
    return info
log.info(f'HG_ROOT: {HG_ROOT}  canali: {len(CHAN_NAMES)}')


## §2 — Feature per-soggetto dai grafi

In [ ]:
# Feature per-soggetto: media di adj (grafo) e H@H.T (ipergrafo) -> triangolo sup.
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_trials = defaultdict(list)
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m: subj_trials[int(m.group(1))].append(p)

CACHE = CKPT_DIR / f'feat_{METRIC}.npz'
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    FEAT_G, FEAT_H, SUBJ = z['feat_g'], z['feat_h'], z['subj'].tolist()
    log.info(f'Cache: G{FEAT_G.shape} H{FEAT_H.shape} ({len(SUBJ)} soggetti)')
else:
    SUBJ, feat_g, feat_h = [], [], []
    for sid in tqdm(sorted(subj_trials), desc='soggetti'):
        adjs, hhts = [], []
        for p in subj_trials[sid]:
            d = torch.load(p, weights_only=False)
            adj = d['adj'].float().numpy()
            if adj.shape != (N_CHAN, N_CHAN): continue
            H = d['H'].float().numpy()
            adjs.append(adj); hhts.append(H @ H.T)
        if not adjs: continue
        SUBJ.append(sid)
        feat_g.append(np.mean(adjs, axis=0)[triu_idx])
        feat_h.append(np.mean(hhts, axis=0)[triu_idx])
    FEAT_G, FEAT_H = np.array(feat_g), np.array(feat_h)
    np.savez(CACHE, feat_g=FEAT_G, feat_h=FEAT_H, subj=np.array(SUBJ))
    log.info(f'Salvato cache: G{FEAT_G.shape} H{FEAT_H.shape} ({len(SUBJ)} soggetti)')


## §3 — Clustering k=2 + silhouette + ARI

In [ ]:
# Clustering k=2 su feature GRAFO e IPERGRAFO + silhouette + ARI cross-metodo
def cluster_k2(feat):
    X = StandardScaler().fit_transform(feat)
    Z = PCA(n_components=min(20, feat.shape[0]-1), random_state=RANDOM_SEED).fit_transform(X)
    lab = KMeans(n_clusters=2, n_init=30, random_state=RANDOM_SEED).fit_predict(Z)
    return lab, silhouette_score(Z, lab)

LAB_G, sil_g = cluster_k2(FEAT_G)
LAB_H, sil_h = cluster_k2(FEAT_H)
ari_gh = adjusted_rand_score(LAB_G, LAB_H)
log.info(f'Grafo:     k=2  silhouette={sil_g:.3f}  sizes={dict(zip(*np.unique(LAB_G, return_counts=True)))}')
log.info(f'Ipergrafo: k=2  silhouette={sil_h:.3f}  sizes={dict(zip(*np.unique(LAB_H, return_counts=True)))}')
log.info(f'ARI grafo vs ipergrafo: {ari_gh:.3f}')

# usa il clustering grafo come riferimento (come EEG_16b)
LAB = LAB_G
# orienta: cluster 0 = quello piu numeroso (convenzione)
if (LAB == 1).sum() > (LAB == 0).sum(): LAB = 1 - LAB
print(f'C0: {(LAB==0).sum()} soggetti   C1: {(LAB==1).sum()} soggetti')


## §4 — Mappa differenza C1−C0 + hub

In [ ]:
# Mappa differenza C1-C0: node strength per elettrodo + hub
def vec_to_sym(v):
    M = np.zeros((N_CHAN, N_CHAN)); M[triu_idx] = v; return M + M.T
adj0 = np.mean([vec_to_sym(FEAT_G[i]) for i in range(len(SUBJ)) if LAB[i]==0], axis=0)
adj1 = np.mean([vec_to_sym(FEAT_G[i]) for i in range(len(SUBJ)) if LAB[i]==1], axis=0)
ns0, ns1 = adj0.sum(1)/(N_CHAN-1), adj1.sum(1)/(N_CHAN-1)
diff = ns1 - ns0

info = make_topo_info()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle(f'V5W_05 — Replica fenotipi (5words, N={len(SUBJ)})\n'
             f'silhouette grafo={sil_g:.3f}  ARI grafo-vs-ipergrafo={ari_gh:.3f}', fontweight='bold')
for ax,(v,ttl,cm) in zip(axes,[(ns0,'C0 node strength','viridis'),(ns1,'C1 node strength','viridis'),(diff,'C1 - C0','RdBu_r')]):
    lim = (None,None) if cm=='viridis' else (-np.abs(diff).max(), np.abs(diff).max())
    im,_ = mne.viz.plot_topomap(v, info, axes=ax, show=False, cmap=cm,
                                vlim=lim if cm=='RdBu_r' else (min(ns0.min(),ns1.min()),max(ns0.max(),ns1.max())),
                                contours=4, sensors=True)
    ax.set_title(ttl, fontsize=11, fontweight='bold'); plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w05_phenotype_diff.png', dpi=160, bbox_inches='tight'); plt.show()

topC1 = [CHAN_NAMES[i] for i in np.argsort(-diff)[:6]]
topC0 = [CHAN_NAMES[i] for i in np.argsort(diff)[:6]]
print(f'Hub C1 (node strength piu alto): {topC1}')
print(f'Hub C0 (node strength piu alto): {topC0}')
print('Tesi: C0 fronto-motor F2/C2/FC2 | C1 fronto-occipital F3/PO8 — confronta i pattern sopra.')
np.savez(CKPT_DIR/'cluster_labels.npz', subj=np.array(SUBJ), labels=LAB)


## §5 — Cross-metrica (PLV, opzionale)

In [ ]:
# (Opzionale) Cross-metrica: se i grafi PLV esistono, ARI con lo split abs_pcc
PLV_ROOT = project_root / 'data' / '5words_subjects' / 'graphs' / 'hypergraphs_pruned_plv'
if PLV_ROOT.exists():
    subj_plv = defaultdict(list)
    for p in sorted(PLV_ROOT.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if m: subj_plv[int(m.group(1))].append(p)
    fg = []; sj = []
    for sid in sorted(subj_plv):
        adjs = []
        for p in subj_plv[sid]:
            d = torch.load(p, weights_only=False); a = d['adj'].float().numpy()
            if a.shape == (N_CHAN, N_CHAN): adjs.append(a)
        if adjs: sj.append(sid); fg.append(np.mean(adjs, axis=0)[triu_idx])
    LAB_PLV, _ = cluster_k2(np.array(fg))
    common = [s for s in SUBJ if s in sj]
    lg = np.array([LAB[SUBJ.index(s)] for s in common])
    lp = np.array([LAB_PLV[sj.index(s)] for s in common])
    print(f'Cross-metrica ARI(abs_pcc, PLV) su {len(common)} soggetti: {adjusted_rand_score(lg, lp):.3f}')
else:
    print('Grafi PLV non presenti — cross-metrica saltata (V5W_02 ha costruito solo abs_pcc).')
    print('Per la validazione cross-metrica, abilita PLV nei METRICS di V5W_02.')


## §6 — Verdetto replica

In [ ]:
# Verdetto replica
print('='*55)
print(f'  V5W_05 — Replica fenotipi su {len(SUBJ)} soggetti 5words')
print('='*55)
print(f'  silhouette k=2 (grafo):      {sil_g:.3f}')
print(f'  silhouette k=2 (ipergrafo):  {sil_h:.3f}')
print(f'  ARI grafo vs ipergrafo:      {ari_gh:.3f}')
print(f'  C0={int((LAB==0).sum())}  C1={int((LAB==1).sum())}')
print('-'*55)
print('  Replica OK se: silhouette > ~0.25 (struttura a 2 gruppi),')
print('  ARI grafo-ipergrafo alto (consistenza), e hub C0/C1 coerenti')
print('  col pattern fronto-motor / fronto-occipital della tesi.')
print('='*55)


## §7 — Permutation test: i 2 gruppi sono reali?

La silhouette bassa (~0.16) non basta: KMeans restituisce sempre 2 cluster.
Test: si permuta ogni feature indipendentemente tra i soggetti (rompe la
struttura congiunta che genera i cluster, conserva le marginali), si rifa
PCA+KMeans+silhouette -> distribuzione nulla. p = frazione di null >= osservato.
Se p < 0.05 la struttura a 2 gruppi e significativa; altrimenti i fenotipi NON
replicano su questa coorte.

In [ ]:
# §7 — Permutation test sulla silhouette k=2 (struttura reale?)
def _silhouette_k2(feat):
    X = StandardScaler().fit_transform(feat)
    Z = PCA(n_components=min(20, feat.shape[0]-1), random_state=RANDOM_SEED).fit_transform(X)
    lab = KMeans(n_clusters=2, n_init=20, random_state=RANDOM_SEED).fit_predict(Z)
    return silhouette_score(Z, lab)

def perm_test(feat, n_perm=1000, seed=42):
    obs = _silhouette_k2(feat)
    rng = np.random.default_rng(seed)
    n, p = feat.shape
    null = np.empty(n_perm)
    for i in range(n_perm):
        idx = rng.random((n, p)).argsort(axis=0)        # permutazione per-colonna
        null[i] = _silhouette_k2(np.take_along_axis(feat, idx, axis=0))
    return obs, null, (null >= obs).mean()

print('Permutation test (1000 perm)...')
obs_g, null_g, p_g = perm_test(FEAT_G)
obs_h, null_h, p_h = perm_test(FEAT_H)
print(f'  GRAFO     : silhouette={obs_g:.3f}  null={null_g.mean():.3f}+/-{null_g.std():.3f}  p={p_g:.3f}')
print(f'  IPERGRAFO : silhouette={obs_h:.3f}  null={null_h.mean():.3f}+/-{null_h.std():.3f}  p={p_h:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (obs, null, pv, ttl) in zip(axes, [(obs_g, null_g, p_g, 'Grafo'), (obs_h, null_h, p_h, 'Ipergrafo')]):
    ax.hist(null, bins=30, color='0.7', edgecolor='white')
    ax.axvline(obs, color='#d62728', lw=2.5, label=f'osservato={obs:.3f}')
    ax.axvline(np.percentile(null, 95), color='k', ls='--', lw=1.2, label='null p95')
    ax.set_title(f'{ttl} - p={pv:.3f}', fontweight='bold')
    ax.set_xlabel('silhouette k=2'); ax.legend(fontsize=8)
fig.suptitle('V5W_05 - Permutation test: struttura a 2 gruppi vs null', fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w05_permutation_test.png', dpi=160, bbox_inches='tight'); plt.show()

print('='*55)
sig = 'SIGNIFICATIVA' if p_g < 0.05 else 'NON significativa'
print(f'  Struttura a 2 gruppi (grafo): {sig}  (p={p_g:.3f})')
print('  p<0.05 -> i fenotipi replicano; altrimenti la struttura')
print('  non si distingue dal caso su questa coorte 5words.')
print('='*55)
